## Walmart Sales EDA
Olá! Seja bem-vindo(a)!

Este projeto foi desenvolvido com foco no desenvolvimento do meu portfólio de análise de dados, demonstrando minhas habilidades em Limpeza e Análise Exploratória de Dados com Python.

### Contexto
O dataset utilizado é o "Walmart Dataset", disponível no Kaggle (https://www.kaggle.com/datasets/yasserh/walmart-dataset).
Ele contém dados históricos de vendas semanais de 45 lojas do Walmart entre 02/2010 e 11/2012.

### Perguntas de Negócio
Esta análise exploratória busca responder 4 perguntas de negócio, sobre o comportamento de vendas do Walmart:

1. Quais lojas apresentam o maior e menor volume de vendas semanais? <br>
   **Objetivo**: Identificar lojas de alto e baixo desempenho para replicar boas práticas ou investigar gargalos operacionais.

2. Lojas com vendas mais baixas estão concentradas em regiões com CPI mais alto? <br>
   **Objetivo**: Verificar se o custo de vida da região explica a baixa performance de certas lojas, ajudando o negócio a diferenciar problemas operacionais de problemas estruturais do mercado local.

3. Semanas com feriados realmente geram vendas maiores do que semanas normais? <br>
   **Objetivo**: Justificar ou redirecionar investimentos em estoque e equipe conforme o real impacto dos feriados nas vendas.

4. O preço do combustível tem correlação com o volume de vendas semanais? <br>
   **Objetivo**: Avaliar se custos de deslocamento afetam o comportamento de compra dos clientes, orientando decisões sobre frete, delivery e localização de lojas.

### Etapas da Análise

- Importação de bibliotecas e carregamento dos dados
- Limpeza, tratamento e preparação
- Análise Exploratória de Dados (EDA) 
- Conclusões

---

Fique à vontade para entrar em contato comigo. Boa leitura!

### Importação das bibliotecas

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
import plotly.express as px



c:\Users\diogo\anaconda3\envs\env-novoprojeto\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Importação de dados
Importação direto da fonte de dados Kaggle, via código.

In [2]:
path = kagglehub.dataset_download("yasserh/walmart-dataset")

In [3]:
df = pd.read_csv(path + '\\Walmart.csv')
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


### 🧹Limpeza, tratamento e exploração
    1. Ajuste tipo da coluna Store de int64 para str, afinal não serão utilizados cálculos.
    2. Ajuste tipo da coluna Date de int64 para data.
    3. Não identificadas linhas vazias para tratamento.
    4. Não identificadas linhas duplicadas para tratamento.

In [20]:
#1
df['Store'] = df['Store'].astype(str)
#2
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         6435 non-null   object        
 1   Date          6435 non-null   datetime64[ns]
 2   Weekly_Sales  6435 non-null   float64       
 3   Holiday_Flag  6435 non-null   int64         
 4   Temperature   6435 non-null   float64       
 5   Fuel_Price    6435 non-null   float64       
 6   CPI           6435 non-null   float64       
 7   Unemployment  6435 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 402.3+ KB


In [5]:
#3
df.isnull().sum()

Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

In [6]:
#4
df.duplicated().sum()

np.int64(0)

### 📊 Exploratória de dados (EDA)
O foco aqui é trazermos analises que responderão as questões de negócio.
    
    1. Quais lojas apresentam o maior e menor volume de vendas semanais?

In [38]:
df_store = df.groupby('Store')['Weekly_Sales'].sum().reset_index()

fig = px.bar(
    df_store,
    x="Store",
    y="Weekly_Sales",
    color="Weekly_Sales",        # cor baseada no valor de Y
    color_continuous_scale="Blues",  # degradê de azul
    title="Sales by Store",
    labels={"Store": "Store", "Weekly_Sales": "Sales"},
    category_orders={"Store": df_store["Store"].tolist()}

)
fig.update_xaxes(tickmode='linear')
fig.show()

Utilizando o Coeficiente de Variação (CV), busca-se identificar se as lojas com maior faturamento possuem uma grande dispersão nos dados, o que poderia indicar que o alto faturamento é resultado de vendas esporádicas, e não de um histórico consistente de

- CV < 15% → baixa variabilidade, vendas consistentes
- CV entre 15% e 30% → variabilidade moderada
- CV > 30% → alta variabilidade, vendas instáveis

In [59]:
lojas = df[df['Store'].isin(['2','4','10','13','14','20'])]

variancia = lojas.groupby('Store')['Weekly_Sales'].std() / lojas.groupby('Store')['Weekly_Sales'].mean()*100
variancia.head()

Store
10    15.913349
13    13.251363
14    15.713674
2     12.342388
20    13.090269
Name: Weekly_Sales, dtype: float64

A suspeita de que o coeficiente de variação seria alto entre as lojas com maiores vendas se mostra infundada, uma vez que a variância oscila entre 12% e 15%, valor similar ao das demais lojas.
Dessa forma, essas lojas demonstram consistência em suas altas vendas, não dependendo apenas de picos pon

In [63]:
variancia2 = df.groupby('Store')['Weekly_Sales'].std() / df.groupby('Store')['Weekly_Sales'].mean()*100
variancia2

Store
1     10.029212
10    15.913349
11    12.226183
12    13.792532
13    13.251363
14    15.713674
15    19.338399
16    16.518065
17    12.552067
18    16.284550
19    13.268012
2     12.342388
20    13.090269
21    17.029239
22    15.678288
23    17.972115
24    12.363738
25    15.986040
26    11.011066
27    13.515544
28    13.732974
29    18.374247
3     11.502141
30     5.200804
31     9.016105
32    11.831049
33     9.286835
34    10.822524
35    22.968111
36    16.257891
37     4.208412
38    11.087545
39    14.990779
4     12.708254
40    12.342978
41    14.817711
42     9.033533
43     6.410363
44     8.179331
45    16.561273
5     11.866844
6     13.582286
7     19.730469
8     11.695283
9     12.689547
Name: Weekly_Sales, dtype: float64